In [ ]:
# Section 0 — Setup & sys.path
import sys
from pathlib import Path

ROOT = Path.cwd().parent          # notebooks/ -> repo root
sys.path.insert(0, str(ROOT / "src" / "rules"))
sys.path.insert(0, str(ROOT / "src" / "ml"))

%matplotlib inline
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Section 1 — Environment

Print library versions used in this pipeline.

In [ ]:
from importlib.metadata import version, PackageNotFoundError

PACKAGES = ["pandas", "numpy", "scikit-learn", "xgboost", "shap", "matplotlib", "scipy", "openpyxl"]
for pkg in PACKAGES:
    try:
        print(f"{pkg:20s} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:20s} NOT INSTALLED")

## Section 2 — Data loading & rail validation

Load the dataset and validate per-rail field coherence.

In [ ]:
import pandas as pd
import numpy as np
import alerts_engine as ae

txs, kyc, mer = ae.load_data()

print("Shapes:")
print(f"  txs : {txs.shape}")
print(f"  kyc : {kyc.shape}")
print(f"  mer : {mer.shape}")
print()
print("Transaction type distribution:")
print(txs["transaction_type"].value_counts().to_string())
print()

# Rail validation
pix_missing_flow  = txs[(txs.transaction_type == "PIX")  & txs.pix_flow.isna()]
card_missing_3ds  = txs[(txs.transaction_type == "Card") & txs.auth_3ds.isna()]
wire_missing_cb   = txs[(txs.transaction_type == "Wire") & txs.cross_border.isna()]

print("Rail validation checks:")
print(f"  PIX without pix_flow     : {len(pix_missing_flow):>4} rows")
print(f"  Card without auth_3ds    : {len(card_missing_3ds):>4} rows")
print(f"  Wire without cross_border: {len(wire_missing_cb):>4} rows")

## Section 3 — Exploratory analysis

Top customers by income mismatch, velocity, and PIX passthrough.

In [ ]:
import matplotlib.pyplot as plt

cust_txs = txs[txs.sender_entity_type == "customer"].copy()
kyc_idx  = kyc.set_index("customer_id")

# ── Income mismatch: outflow vs monthly income ────────────────────────────────
outflow = cust_txs.groupby("sender_id")["amount_brl"].sum().rename("total_outflow")
monthly = (kyc_idx["annual_income_brl"] / 12).rename("monthly_income")
mismatch = pd.concat([outflow, monthly], axis=1).dropna()
mismatch["income_multiple"] = mismatch["total_outflow"] / mismatch["monthly_income"]
mismatch["declared_occupation"] = kyc_idx["declared_occupation"].reindex(mismatch.index)
top_mismatch = mismatch.sort_values("income_multiple", ascending=False).head(10)

print("Top 10 customers by outflow / monthly income:")
display(top_mismatch[["total_outflow", "monthly_income", "income_multiple", "declared_occupation"]]
        .style.format({"total_outflow": "R${:,.0f}", "monthly_income": "R${:,.0f}", "income_multiple": "{:.1f}x"}))
print()

# ── Velocity: max transactions in a single day ────────────────────────────────
cust_txs["day"] = pd.to_datetime(cust_txs["timestamp"]).dt.date
daily_vel = (cust_txs.groupby(["sender_id", "day"]).size()
                     .groupby("sender_id").max()
                     .rename("max_txs_in_a_day")
                     .sort_values(ascending=False)
                     .head(10))

print("Top 10 customers by max transactions in a single day:")
display(daily_vel.to_frame())
print()

# ── PIX passthrough: pix_out / pix_in ────────────────────────────────────────
pix_out = (txs[(txs.pix_flow == "cash_out") & (txs.sender_entity_type == "customer")]
              .groupby("sender_id")["amount_brl"].sum().rename("pix_out"))
pix_in  = (txs[txs.pix_flow == "cash_in"]
              .groupby("receiver_id")["amount_brl"].sum().rename("pix_in"))
passthru = pd.concat([pix_out, pix_in.reindex(pix_out.index)], axis=1).fillna(0)
passthru["pix_in"] = passthru["pix_in"].replace(0, np.nan)
passthru["passthru_ratio"] = passthru["pix_out"] / passthru["pix_in"]
top_passthru = (passthru[passthru["pix_out"] >= 20_000]
                .sort_values("passthru_ratio", ascending=False)
                .head(10))

print("Top 10 PIX passthrough customers (pix_out >= R$20k, sorted by ratio):")
display(top_passthru.style.format({"pix_out": "R${:,.0f}", "pix_in": "R${:,.0f}", "passthru_ratio": "{:.1f}x"}))

## Section 4 — Rules engine

Run the Phase 2 rules engine inline and inspect output.

In [ ]:
fires = ae.run_rules(txs, kyc, mer)
per   = ae.score(fires)

# Build alerts DataFrame (mirrors alerts_engine.main())
rows = [{
    "customer_id":     cid,
    "triggered_rules": "|".join(sorted(set(e["rules"]))),
    "total_score":     e["score"],
    "escalation_band": ae.assign_band(e),
    "hard_alert_flag": e["hard"],
} for cid, e in per.items()]

alerts = (pd.DataFrame(rows)
            .sort_values(["total_score", "hard_alert_flag"], ascending=[False, False])
            .reset_index(drop=True))

print("Escalation band distribution:")
band_counts = alerts["escalation_band"].value_counts()
print(band_counts.to_string())
print()

print("Top 15 scored customers:")
display(alerts.head(15))

# Bar chart of band counts
fig, ax = plt.subplots(figsize=(8, 4))
band_counts.sort_index().plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Customers by escalation band (Phase 2 rules engine)", fontsize=13)
ax.set_xlabel("Escalation band")
ax.set_ylabel("Customer count")
ax.tick_params(axis="x", rotation=25)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height() + 1),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

## Section 5 — Isolation Forest: Independent Anomaly Detection

### The weak-label circularity problem and the v2 ML response

The Phase 3 XGBoost model derives its labels from the Phase 2 rules engine, which creates a structural circularity. The v2 pipeline addresses this in two layered ways:

1. **Target narrowing.** The XGBoost target is no longer the full composite score. It is `behavioral_risk_score`, the sum of weights for the **soft behavioral** core rules only — R02 structuring, R03 income mismatch, R09 PEP. Hard regulatory alerts (R08 sanctions, R16 self-merchant, R21 network linkage) are deliberately **excluded** from the target: those are binary regulatory facts owned by the rules engine, not patterns ML should be asked to predict from transactional behavior. The remaining 15 rules are held out and survive only as raw aggregates the model must rediscover.
2. **Independent unsupervised signal.** Even with the narrowed target, label–feature proximity persists. `src/ml/isolation_forest.py` adds a fully unsupervised **Isolation Forest** trained on raw transaction features only — no rules scores, no labels, no engine output. It feeds the v2 pipeline two ways: as its own ranked output (`isolation_forest_scores.csv`) for direct divergence analysis, and as a single feature column (`iforest_anomaly_score`) inside the XGBoost feature matrix.

The nine raw transaction features the IF uses:

| Feature | Signal |
|---|---|
| `amount_brl` | Transaction magnitude |
| `tx_hour` | Time-of-day behaviour |
| `mcc_risk_enc` | Merchant category risk (encoded) |
| `country_risk_geo_enc` | Geographic risk (encoded) |
| `device_rooted_enc` | Rooted device indicator |
| `anon_enc` | Anonymisation type: None / VPN+Proxy / Tor |
| `tx_type_enc` | Rail: PIX / Card / Wire |
| `card_present_enc` | Card-not-present indicator |
| `auth_3ds_enc` | Missing 3DS auth indicator |

Aggregation: per-customer mean across all transactions.

**Three independent signals therefore feed the analyst queue:** rules engine (regulatory facts), XGBoost regressor (behavioral patterns), Isolation Forest (unsupervised anomaly). Convergence across layers strengthens confidence; divergence is informative.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src' / 'ml'))
import isolation_forest as iso_mod
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display as ipy_display, Image as IpyImage
import io

# ── Run Isolation Forest ──────────────────────────────────────────────────────
if_scores, _ = iso_mod.run(txs, mer)

print(f"Customers scored: {len(if_scores)}")
print(f"Flagged as anomalous (top 5%): {if_scores.is_anomaly.sum()}")
print(f"\nTop 20 most anomalous customers:")
display(if_scores.head(20))

# ── Scatter: IF score vs XGBoost probability ──────────────────────────────────
ml_out = pd.read_csv(ROOT / 'outputs' / 'rankings' / 'ml_ranked_output.csv')

merged = if_scores.merge(
    ml_out[['customer_id', 'predicted_probability', 'predicted_band']],
    on='customer_id', how='inner'
)

# Focus on the top cohort: ML Tier 1/2/3 and top-5% IF anomalies
top_cohort = merged[
    merged.predicted_band.isin(['ML Tier 1', 'ML Tier 2', 'ML Tier 3']) |
    merged.is_anomaly
].copy()

band_colors = {
    'ML Tier 1': '#d62728',
    'ML Tier 2': '#ff7f0e',
    'ML Tier 3': '#aec7e8',
    'Routine':   '#dddddd',
}

fig, ax = plt.subplots(figsize=(9, 6))

for band, grp in top_cohort.groupby('predicted_band'):
    c = band_colors.get(band, '#cccccc')
    ax.scatter(grp.anomaly_score, grp.predicted_probability,
               c=c, s=20, alpha=0.75, label=band, zorder=2)

# Highlight C102290
subj = merged[merged.customer_id == 'C102290']
ax.scatter(subj.anomaly_score, subj.predicted_probability,
           c='black', s=120, zorder=5, marker='*')
ax.annotate(
    'C102290\n(XGB rank #2, IF rank #1410)',
    xy=(float(subj.anomaly_score.iloc[0]), float(subj.predicted_probability.iloc[0])),
    xytext=(float(subj.anomaly_score.iloc[0]) + 0.012, float(subj.predicted_probability.iloc[0]) - 0.012),
    fontsize=8, color='black',
    arrowprops=dict(arrowstyle='->', color='black', lw=0.8),
    bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='black', alpha=0.85),
)

ax.set_xlabel('Isolation Forest anomaly score (higher = more anomalous)', fontsize=10)
ax.set_ylabel('XGBoost predicted probability', fontsize=10)
ax.set_title(
    'IF anomaly score vs XGBoost probability — top cohort\n'
    'Agreement (upper-right) strengthens escalation confidence; '
    'divergence signals rule-coverage gaps',
    fontsize=10
)

handles = [mpatches.Patch(color=v, label=k) for k, v in band_colors.items() if k != 'Routine']
handles.append(plt.Line2D([0],[0], marker='*', color='w',
               markerfacecolor='black', markersize=11, label='C102290 (showcase)'))
ax.legend(handles=handles, fontsize=8, loc='lower right')
ax.grid(True, alpha=0.2, linewidth=0.5)
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
buf.seek(0)
ipy_display(IpyImage(data=buf.read(), format='png'))
plt.close(fig)

# Summary: where does C102290 sit in each ranking?
xgb_rank  = int(ml_out.reset_index().query("customer_id == 'C102290'").index[0]) + 1
if_rank   = int(if_scores[if_scores.customer_id == 'C102290']['rank'].values[0])
print(f"\nC102290 rankings:")
print(f"  XGBoost probability rank : {xgb_rank} / 2500")
print(f"  Isolation Forest rank     : {if_rank} / 2500")
print(f"  Interpretation: XGBoost flags C102290 based on rule-correlated features;")
print(f"  IF ranks it mid-population, suggesting the risk comes from composite")
print(f"  rule convergence rather than raw transaction-level anomaly.")


## Phase 3 — ML Prioritization Layer (v2)

**Why XGBoost regression on `behavioral_risk_score`:** v1 trained a binary classifier on `composite_score ≥ 13 OR hard_alert`. Two problems: (1) labels-as-features-thresholded inflated PR-AUC to 0.998, dominated by leakage; (2) 1,726 of 2,500 customers fell in the score-4-to-12 grey zone and were excluded from training entirely.

v2 reframes the task as **regression on a continuous behavioral-soft-rules target** (R02 structuring + R03 income mismatch + R09 PEP, sum of weights ∈ {0,…,9}). Hard alerts (R08 sanctions / R16 self-merchant / R21 network linkage) are excluded — those are owned by the rules engine, not ML. All 2,500 customers participate; `pep_flag` is dropped (1:1 mechanical mapping to R09); five new features are added (Isolation Forest anomaly score, four counterparty/merchant exposure features). Isotonic calibration maps the regression output to `P(top decile composite_score OR hard_alert)`.

**Test metrics (n = 750):** RMSE 0.77, MAE 0.55, R² 0.70, Spearman 0.85.
**Full-population Spearman (n = 2,500) vs target:** 0.87.

**On the v1-labeled subset (apples-to-apples):** PR-AUC drops from 0.998 to 0.957, ROC-AUC from 0.995 to 0.893. The 4-point PR-AUC drop is the cost of admitting all 2,500 customers and stripping the most leak-prone features — not a regression in capability.

**Cohort behavior under v2:** Phase 1 subjects whose Phase 2 priority is driven by hard regulatory facts (C100091 sanctions, C101028 sanctions, C101582 sanctions, C101445 network) correctly drop to low ML probability — the model can no longer peek at those signals — while the rules engine still escalates them via `hard_alert = True`. Phase 1 subjects driven by behavioral signals (C102290, C100837, C102093) stay at the top of the ML queue, driven by genuine behavioral features. This is the layered behavior v2 was designed to produce.

In [ ]:
import ml_pipeline as mp
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score,
    roc_curve, precision_recall_curve
)
from scipy.stats import spearmanr
from IPython.display import Image
import numpy as np

# Filter to feature/label window
txs["timestamp"] = pd.to_datetime(txs["timestamp"])
print(f"Feature/label window: {mp.FEAT_START.date()} → {(mp.FEAT_END - pd.Timedelta(days=1)).date()}")

# ── Run the v2 pipeline end-to-end (this is the same call detection_agent uses) ──
print("\nRunning v2 pipeline: build_targets + build_features + train_regressor + calibrate + SHAP...")
result = mp.score_population(txs, kyc, mer, with_shap=True)

ranked    = result['ranked']
cuts      = result['cuts']
model     = result['model']
X         = result['feature_matrix']
shap_vals = result['shap_values']
targets   = result['targets']
X_test    = result['X_test']
y_test    = result['y_test']

y_behav   = targets['behavioral_risk_score'].astype(float).reindex(X.index)
composite = targets['composite_score'].astype(float).reindex(X.index)

print(f"\nFeature matrix: {X.shape[0]} customers × {X.shape[1]} features")
print(f"behavioral_risk_score: mean={y_behav.mean():.2f}, std={y_behav.std():.2f}, max={int(y_behav.max())}")

# ── Regression metrics on test ──
test_pred = model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
mae  = float(mean_absolute_error(y_test, test_pred))
r2   = float(r2_score(y_test, test_pred))
rho_test, _ = spearmanr(test_pred, y_test)
print(f"\nRegression metrics (test set, n={len(y_test)}):")
print(f"  RMSE      {rmse:.4f}")
print(f"  MAE       {mae:.4f}")
print(f"  R²        {r2:.4f}")
print(f"  Spearman  {rho_test:.4f}")

# ── Binary metrics on v1-labeled test subset (apples-to-apples vs v1) ──
prob_full = ranked.set_index('customer_id')['predicted_probability']
y_v1 = composite.apply(lambda s: 1 if s >= 13 else (0 if s <= 4 else np.nan))
y_v1[targets['hard_alert'].reindex(X.index)] = 1
test_v1 = y_v1.reindex(X_test.index).dropna()
prob_test = prob_full.reindex(test_v1.index)
pr_auc = average_precision_score(test_v1, prob_test)
roc    = roc_auc_score(test_v1, prob_test)
print(f"\nBinary metrics on v1-labeled test subset (n={len(test_v1)}, pos={int(test_v1.sum())}):")
print(f"  PR-AUC      {pr_auc:.4f}    (v1: 0.998)")
print(f"  ROC-AUC     {roc:.4f}    (v1: 0.995)")

# ── ROC and PR curves side-by-side (on v1-labeled subset for comparability) ──
fpr, tpr, _  = roc_curve(test_v1, prob_test)
prec, rec, _ = precision_recall_curve(test_v1, prob_test)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr, color='#1f77b4', lw=2)
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title(f'ROC — v1-labeled subset (AUC={roc:.4f})')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rec, prec, color='#d62728', lw=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall (PR-AUC={pr_auc:.4f})')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Display the ranked output
print(f"\nTop 10 by predicted_probability:")
display(ranked.head(10))

## Section 6 — C102290 end-to-end case

End-to-end review of C102290 — the primary SAR showcase subject.

In [ ]:
SUBJECT = "C102290"

# ── Rules fired ───────────────────────────────────────────────────────────────
subj_alert = alerts[alerts.customer_id == SUBJECT]
print(f"Rules fired for {SUBJECT}:")
display(subj_alert[["customer_id", "triggered_rules", "total_score", "escalation_band", "hard_alert_flag"]])
print()

# ── ML probability and band (from saved output) ───────────────────────────────
ml_out  = pd.read_csv(ROOT / "outputs" / "rankings" / "ml_ranked_output.csv")
subj_ml = ml_out[ml_out.customer_id == SUBJECT]
print(f"ML output for {SUBJECT}:")
display(subj_ml[["customer_id", "predicted_probability", "predicted_band",
                  "actual_label", "top_shap_driver_1", "top_shap_driver_2", "top_shap_driver_3"]])
print()

# ── Key metrics summary ───────────────────────────────────────────────────────
kyc_row  = kyc_idx.loc[SUBJECT]
subj_txs = cust_txs[cust_txs.sender_id == SUBJECT]

declared_annual  = 11_177.0
declared_monthly = declared_annual / 12
total_outflow    = subj_txs["amount_brl"].sum()
income_multiple  = total_outflow / declared_monthly

# PIX passthrough
subj_pix_out = txs[(txs.pix_flow == "cash_out")  & (txs.sender_id == SUBJECT)]["amount_brl"].sum()
subj_pix_in  = txs[(txs.pix_flow == "cash_in")   & (txs.receiver_id == SUBJECT)]["amount_brl"].sum()
pix_passthru_pct = (subj_pix_out / subj_pix_in * 100) if subj_pix_in > 0 else float("inf")

# Anonymization events
anon_events = int(subj_txs[subj_txs.ip_proxy_vpn_tor.isin(["Tor", "VPN", "Proxy"])].shape[0])

summary = {
    "Declared annual income (R$)":        f"{declared_annual:,.0f}",
    "Monthly income (R$)":                f"{declared_monthly:,.0f}",
    "Total outflow (R$)":                 f"{total_outflow:,.0f}",
    "Income multiple":                    f"{income_multiple:.0f}x",
    "PIX passthrough (%)": f"{pix_passthru_pct:,.0f}%",
    "Anonymization events (Tor/VPN/Proxy)": anon_events,
    "KYC risk score":                     kyc_row["kyc_risk_score"],
    "PEP status":                         kyc_row["pep"],
}

print(f"Key risk metrics — {SUBJECT}:")
display(pd.DataFrame.from_dict(summary, orient="index", columns=["Value"]))
print()

# ── First 60 lines of SAR ─────────────────────────────────────────────────────
sar_path  = ROOT / "docs" / "phase1" / "SAR-2025-C102290-01.md"
sar_lines = sar_path.read_text().splitlines()[:60]
print(f"\n--- {sar_path.name} (first 60 lines) ---\n")
print("\n".join(sar_lines))

## Section 7 — Entity Graph: C102290 cluster

### Why merchant convergence is an AML red flag

Merchant convergence occurs when two or more customers who were flagged **independently** — on the basis of their own activity, with no prior link assumed — are found to share the same receiving merchants. This matters for two reasons:

1. **It is not explained by coincidence at scale.** A platform with 1,000 merchants and 2,500 customers has a low baseline probability that two high-risk customers independently route funds to the same obscure merchant. When it happens across multiple merchants simultaneously (here: M200460 and M200186), the probability of independence drops sharply.

2. **It suggests coordinated layering, not parallel anomalies.** In classic layering, multiple placement accounts feed funds upward through shared distribution nodes — merchants, aggregators, or wallets — before integration. The shared merchant acts as a convergence point that links otherwise unconnected sources. This pattern is captured by rule R20 (Merchant Convergence) and elevates both subjects from individual SARs to a potential network SAR.

The graph below shows C102290 and C100880 as independently flagged customers (red), with directed edges to the two shared merchants (blue). Edge width is proportional to transaction amount. Both customers reached these merchants via different rails and on different dates — the convergence is structural, not transactional coincidence.


In [ ]:
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display as ipy_display, Image as IpyImage
import io, base64

# ── Build graph from real transaction data ────────────────────────────────────
subjects  = ['C102290', 'C100880']
merchants = ['M200460', 'M200186']

sub_txs = txs[
    txs.sender_id.isin(subjects) & txs.receiver_id.isin(merchants)
][['sender_id', 'receiver_id', 'amount_brl', 'transaction_type']].copy()

G = nx.DiGraph()
G.add_nodes_from(subjects,  node_type='customer')
G.add_nodes_from(merchants, node_type='merchant')

for _, row in sub_txs.iterrows():
    G.add_edge(
        row['sender_id'], row['receiver_id'],
        amount=row['amount_brl'],
        rail=row['transaction_type'],
    )

# ── Layout ────────────────────────────────────────────────────────────────────
pos = {
    'C102290': (0.0,  0.6),
    'C100880': (0.0, -0.6),
    'M200460': (1.0,  0.6),
    'M200186': (1.0, -0.6),
}

node_colors = [
    '#e74c3c' if G.nodes[n]['node_type'] == 'customer' else '#2980b9'
    for n in G.nodes()
]

max_amount = max(d['amount'] for _, _, d in G.edges(data=True))
edge_widths = [3.5 * (d['amount'] / max_amount) + 0.8 for _, _, d in G.edges(data=True)]
edge_labels = {
    (u, v): f"{d['rail']}\nR${d['amount']:,.0f}"
    for u, v, d in G.edges(data=True)
}

# ── Draw ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=2200, ax=ax, alpha=0.92)
nx.draw_networkx_labels(G, pos, font_size=10, font_color='white', font_weight='bold', ax=ax)
nx.draw_networkx_edges(
    G, pos, width=edge_widths, edge_color='#555555',
    arrows=True, arrowsize=18,
    connectionstyle='arc3,rad=0.12', ax=ax,
    node_size=2200,
)
nx.draw_networkx_edge_labels(
    G, pos, edge_labels=edge_labels,
    font_size=8, label_pos=0.38, ax=ax,
    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#cccccc', alpha=0.85),
)

legend = [
    mpatches.Patch(color='#e74c3c', label='Customer (independently flagged)'),
    mpatches.Patch(color='#2980b9', label='Merchant (shared convergence node)'),
]
ax.legend(handles=legend, loc='lower center', fontsize=9, framealpha=0.9)
ax.set_title(
    'Entity graph — C102290 / C100880 merchant convergence (R20)\n'
    'Edge width ∝ transaction amount · edges from real transaction data',
    fontsize=11,
)
ax.axis('off')
plt.tight_layout()

# Embed PNG as base64 so it appears in the notebook output
buf = io.BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
buf.seek(0)
ipy_display(IpyImage(data=buf.read(), format='png'))
plt.close(fig)

print('\nEdge summary:')
print(sub_txs.groupby(['sender_id', 'receiver_id']).agg(
    txn_count=('amount_brl', 'size'),
    total_brl=('amount_brl', 'sum'),
    rail=('transaction_type', 'first'),
).rename(columns={'total_brl': 'Total (R$)'}).to_string())
